# MNIST Inference with OpenEye Toolkit and ONNX
This notebook demonstrates how to use the OpenEye Toolkit to run inference 
on the MNIST dataset using an ONNX model.

It covers training a simple neural network, exporting it to ONNX format, and
then quantizing it using the ONNX Runtime and the OpenEye Toolkit to simulate
inference on an image of the test dataset.

In [35]:
# check if required packages are installed, if not ask to install them using pip
import importlib
import subprocess
import sys

required_packages = [
    "torch",
    "torchvision",
    "onnx",
    "onnxruntime",
    "onnxscript",
    "onnxoptimizer"
]

all_packages_installed = True
for package in required_packages:
    if importlib.util.find_spec(package) is None:
        all_packages_installed = False
        print(f"{package} is not installed. Should I install it for you? (y/n)")
        choice = input().lower()
        if choice == 'y':
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        else:
            print(f"Please install {package} manually.")

if all_packages_installed:
    print("All required packages are already installed.")

All required packages are already installed.


In [36]:
# import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from mnist_conv_net import SimpleMNISTConvNet

# additional imports for ONNX handling
import onnx
import onnxruntime as ort
import onnxoptimizer

import numpy as np

In [37]:
from mnist_conv_net import SimpleMNISTConvNet, load_simple_mnist_model

model = SimpleMNISTConvNet()

#load_simple_mnist_model('mnist_unquantized_model.pth', model)

In [38]:
import os
import open_eye

# Falls die Datei im selben Ordner wie dein Notebook liegen SOLLTE:
# Prüfe mit diesem Befehl, was aktuell wirklich in deinem Ordner liegt:
print("Dateien im aktuellen Verzeichnis:", os.listdir('.'))
print (model)

Dateien im aktuellen Verzeichnis: ['parameters.vh', 'simple_mnist_processed.onnx', '.ipynb_checkpoints', 'simple_mnist_convnet_int8.onnx', 'mnist_tensorflow.ipynb', 'mnist_unquantized_model.onnx.data', 'unified_model_loader_demo.ipynb', 'mnist_pytorch.ipynb', 'mnist.ipynb', 'mnist_onnx.ipynb', 'mnist_quantized_model.tflite', '__pycache__', 'mnist_conv_net.py', 'imagenet_pytorch.ipynb', 'simple_mnist_convnet.onnx']
SimpleMNISTConvNet(
  (Conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (Pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (Pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (FC1): Linear(in_features=3136, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [39]:
model.eval()
dummy_input = torch.randn(1, 1, 28, 28)
onnx_file_path = "simple_mnist_convnet.onnx"

# Wir deaktivieren dynamo explizit und erhöhen das opset auf Version 18,
# wie es von deiner PyTorch-Version gewünscht wird.
torch.onnx.export(
    model,
    dummy_input,
    onnx_file_path,
    export_params=True,
    opset_version=18,          # Erhöht auf 18, um Warnungen zu vermeiden
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input' : {0 : 'batch_size'},
                  'output' : {0 : 'batch_size'}},
    dynamo=False               # Zwingt PyTorch in den stabilen Legacy-Pfad
)

print(f"Modell erfolgreich nach {onnx_file_path} exportiert!")

Modell erfolgreich nach simple_mnist_convnet.onnx exportiert!


/tmp/ipykernel_2417877/1458454849.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [51]:
import numpy as np
import onnxruntime as ort

# 1. ONNX-Modell laden (es wird eine Inferenz-Session gestartet)
onnx_file_path = "simple_mnist_convnet.onnx"
session = ort.InferenceSession(onnx_file_path)

# 2. Namen der Eingabe- und Ausgabe-Nodes auslesen
# (Wir haben sie beim Export 'input' und 'output' genannt)
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

# 3. Testdaten vorbereiten 
# WICHTIG: ONNX-Runtime erwartet reine NumPy-Arrays, KEINE PyTorch-Tensoren!
# Wir erstellen hier beispielhaft ein "Bild" (bzw. einen Batch von 1 Bild)
dummy_image = np.random.randn(1, 1, 28, 28).astype(np.float32)

# 4. Inferenz ausführen
# Das Modell erwartet ein Dictionary, das den Node-Namen mit den Daten verknüpft
outputs = session.run([output_name], {input_name: dummy_image})

# 5. Ergebnis auswerten
# outputs[0] enthält die Raw-Logits (10 Werte für die 10 MNIST-Ziffern)
logits = outputs[0]
predicted_class = np.argmax(logits, axis=1)

print("Logits des Modells:", logits)
print("Vorhergesagte Ziffer:", predicted_class[0])


Logits des Modells: [[ 0.05665779 -0.00847745 -0.05467002 -0.11052252 -0.141207    0.01590565
  -0.19930242 -0.10076353  0.05390662 -0.02362201]]
Vorhergesagte Ziffer: 0


In [47]:
from onnxruntime.quantization import quant_pre_process, quantize_dynamic, QuantType

# 1. Dateipfade definieren
model_fp32 = "simple_mnist_convnet.onnx"
model_prepared = "simple_mnist_processed.onnx"
model_int8 = "simple_mnist_convnet_int8.onnx"

# 2. Pre-Processing ausführen (Optimiert die Graph-Struktur für die Quantisierung)
quant_pre_process(
    input_model_path=model_fp32,
    output_model_path=model_prepared
)

# 3. Das optimierte Modell in INT8 konvertieren
quantize_dynamic(
    model_input=model_prepared,    # Jetzt nutzen wir das vorbereitete Modell
    model_output=model_int8,
    weight_type=QuantType.QInt8
)

print(f"Perfekt! Das optimierte INT8-Modell wurde unter {model_int8} gespeichert.")

Perfekt! Das optimierte INT8-Modell wurde unter simple_mnist_convnet_int8.onnx gespeichert.


In [71]:
model_path = model_int8
model = onnx.load(model_path)

print(f"--- Kompakte Layer-Übersicht von {model_path} ---\n")

seen_layers = set("")
ordered_layers = []

for node in model.graph.node:
    name = node.name
    
    # Wir isolieren den Hauptnamen (z.B. "conv1" aus "/conv1/Conv_quant")
    if name.startswith('/'):
        # Extrahiert den Namen zwischen den ersten beiden Slashes
        parts = name.split('/')
        main_layer = parts[1] if len(parts) > 1 else name
    else:
        # Falls kein Slash da ist (z.B. bei "input_QuantizeLinear" oder "Node_0")
        main_layer = name.split('_')[0]
        
    # Ein paar Hilfs-Nodes filtern, um es noch sauberer zu machen
    if main_layer in ['Node', 'input', 'output']:
        continue

    if main_layer not in seen_layers:
        seen_layers.add(main_layer)
        ordered_layers.append(main_layer)

# Ausgabe der bereinigten Sequenz
for i, layer in enumerate(ordered_layers):
    print(f"[{i+1}] {layer}")

--- Kompakte Layer-Übersicht von simple_mnist_convnet_int8.onnx ---

[1] 
[2] Conv1
[3] Relu
[4] Pool1
[5] Conv2
[6] Relu_1
[7] Pool2
[8] Conv3
[9] Relu_2
[10] Flatten
[11] Flatten_output_0_QuantizeLinear
[12] FC1


In [70]:
from onnx import numpy_helper

model = onnx.load("simple_mnist_convnet_int8.onnx")

# Wir bauen ein Wörterbuch aller Initializer (Gewichte/Scales) für den schnellen Zugriff
initializers = {init.name: numpy_helper.to_array(init) for init in model.graph.initializer}
print(initializers)
print("--- Extrahierte Gewichte & Parameter für Hardware-Portierung ---\n")

# Wir suchen gezielt nach den Haupt-Layern
for layer_name in ['Conv1', 'Conv2', 'Conv3', 'FC1']:
    print(f"== Layer: {layer_name} ==")
    
    # 1. Quantisierte 8-Bit Gewichte holen
    weight_key = f"{layer_name}.weight_quantized"
    if weight_key in initializers:
        weights = initializers[weight_key]
        print(f"  -> Gewichte (INT8):   Shape {weights.shape} | Typ: {weights.dtype}")
    else:
        # Falls fc1 als klassisches MatMulinteger exportiert wurde
        weight_key_alt = f"{layer_name}.weight"
        if weight_key_alt in initializers:
            weights = initializers[weight_key_alt]
            print(f"  -> Gewichte (Raw):    Shape {weights.shape} | Typ: {weights.dtype}")

    # 2. Skalierungsfaktor (Scale) holen
    scale_key = f"{layer_name}.weight_scale"
    if scale_key in initializers:
        scale = initializers[scale_key]
        print(f"  -> Weight Scale:       {scale}")

    # 3. Nullpunkt (Zero Point) holen
    zp_key = f"{layer_name}.weight_zero_point"
    if zp_key in initializers:
        zp = initializers[zp_key]
        print(f"  -> Weight Zero Point:  {zp}")

    # 4. Bias holen
    bias_key = f"{layer_name}.bias"
    if bias_key in initializers:
        bias = initializers[bias_key]
        print(f"  -> Bias:               Shape {bias.shape} | Typ: {bias.dtype}")
        
    print("-" * 50)

{'Conv1.bias': array([-0.2808553 ,  0.07341202,  0.05300438,  0.06071329, -0.07606276,
        0.0031995 , -0.24607532, -0.26331663,  0.21598566, -0.07286998,
        0.09305398, -0.20262775,  0.24662161,  0.04806626, -0.15815811,
       -0.2436285 ,  0.12504053, -0.0382994 ,  0.08443324, -0.1644758 ,
        0.0150303 ,  0.01325468,  0.03236679, -0.12910375,  0.2369036 ,
        0.08925077,  0.17879789, -0.07693999,  0.26097465,  0.27682984,
        0.29321083,  0.214493  ], dtype=float32), 'Conv2.bias': array([ 0.01080319, -0.02930619,  0.02063729,  0.04783219, -0.02280712,
        0.0034328 ,  0.02018278,  0.05684117,  0.05415611, -0.01707359,
       -0.03237068,  0.02503708, -0.03389039,  0.00530155,  0.02200599,
        0.02301156,  0.05419567, -0.00737009,  0.00774229,  0.01398102,
       -0.00443961, -0.00225027, -0.01981536, -0.01685204, -0.05857116,
       -0.02865953,  0.03393264,  0.00729923, -0.03163091,  0.01917746,
        0.05680516, -0.02866217,  0.04805643, -0.00984504

In [66]:
# Wir nehmen an, du hast die 'initializers' aus der vorherigen Schleife geladen:
#print(initializers)
conv1_weights = initializers['Conv1.weight_quantized']

print("=== Einblick in die echten INT8-Gewichte von conv1 ===")
print("Datentyp:", conv1_weights.dtype)
print("Shape (Out_Channels, In_Channels, Kernel_H, Kernel_W):", conv1_weights.shape)

print("\nKonkrete INT8-Werte des allerersten 3x3 Filters (Kanal 0, Filter 0):")
# Greift auf den ersten Filter zu [Output-Kanal 0, Input-Kanal 0, alle H, alle W]
print(conv1_weights[0, 0, :, :])


=== Einblick in die echten INT8-Gewichte von conv1 ===
Datentyp: int8
Shape (Out_Channels, In_Channels, Kernel_H, Kernel_W): (32, 1, 3, 3)

Konkrete INT8-Werte des allerersten 3x3 Filters (Kanal 0, Filter 0):
[[  82   12  -71]
 [ 111  -29  -34]
 [ 111  -49 -125]]


In [68]:
for name in ['Conv1', 'Conv2', 'Conv3', 'FC1']:
    key = f"{name}.weight_quantized"
    if key in initializers:
        w = initializers[key]
        print(f"Layer {name:5} | Min-Wert: {w.min():4} | Max-Wert: {w.max():4} | Mittelwert: {w.mean():.2f}")

Layer Conv1 | Min-Wert: -127 | Max-Wert:  126 | Mittelwert: -0.07
Layer Conv2 | Min-Wert: -127 | Max-Wert:  127 | Mittelwert: -0.59
Layer Conv3 | Min-Wert: -127 | Max-Wert:  127 | Mittelwert: -0.32
Layer FC1   | Min-Wert: -127 | Max-Wert:  127 | Mittelwert: -0.52


In [55]:
import onnx
from onnx import numpy_helper

def parse_onnx_with_fused_relu(model_path):
    model = onnx.load(model_path)
    initializers = {init.name: numpy_helper.to_array(init) for init in model.graph.initializer}
    
    # 1. Schritt: Wir bauen ein Mapping von [Eingangs-Tensor-Name] -> [Knoten, der ihn liest]
    # Das hilft uns zu sehen, was nach einem Layer passiert.
    next_node_map = {}
    for node in model.graph.node:
        for inp in node.input:
            next_node_map[inp] = node

    hardware_layers = []
    layer_idx = 1
    
    # Wir gehen alle Knoten durch
    for node in model.graph.node:
        # Wir interessieren uns primär für die rechenintensiven Basis-Layer
        if node.op_type not in ['ConvInteger', 'QLinearConv', 'MatMulInteger', 'QLinearMatMul', 'MaxPool']:
            continue
            
        # Standard-Typisierung zuordnen
        if node.op_type in ['ConvInteger', 'QLinearConv']:
            layer_type = 'CONVOLUTION'
        elif node.op_type in ['MatMulInteger', 'QLinearMatMul']:
            layer_type = 'FULLY_CONNECTED'
        elif node.op_type == 'MaxPool':
            layer_type = 'MAX_POOLING'

        # Basis-Struktur aufbauen
        layer_data = {
            "id": layer_idx,
            "type": layer_type,
            "name": node.name,
            "has_relu": False  # Standardmäßig hat das Layer kein ReLU (wie in TF)
        }
        
        # --- LOOK-AHEAD LOGIK FÜR FUSED ACTIVATION ---
        # Jedes Layer hat einen Ausgangs-Tensor (meistens am Index 0)
        output_tensor_name = node.output[0]
        
        # Wir schauen nach, wer diesen Ausgang als nächstes konsumiert
        # Aber Achtung: Manchmal hängen bei INT8 noch "Add" (für Bias) oder "Mul" dazwischen.
        # Wir verfolgen den Pfad im Graphen ein Stück weiter, bis wir ein ReLU finden oder nicht.
        current_tensor = output_tensor_name
        for _ in range(5):  # Maximal 5 Schritte im INT8-Rauschen vorwärts suchen
            if current_tensor in next_node_map:
                next_node = next_node_map[current_tensor]
                if next_node.op_type == 'Relu':
                    layer_data["has_relu"] = True
                    break
                # Weitergehen zum Ausgang des Zwischenknotens (z.B. Cast, Add, Mul)
                current_tensor = next_node.output[0]
            else:
                break
        # ---------------------------------------------

        # Gewichte extrahieren (nur für Conv und Fully Connected)
        if layer_type in ['CONVOLUTION', 'FULLY_CONNECTED']:
            weight_name = node.input[1]
            if weight_name in initializers:
                weights = initializers[weight_name]
                layer_data["weights"] = {
                    "shape": list(weights.shape),
                    "dtype": str(weights.dtype),
                    "raw_values": weights.flatten().tolist()
                }
            
            # Skalierungsfaktoren und Biases einsammeln
            for inp in node.input:
                if 'scale' in inp and inp in initializers:
                    layer_data["scale"] = float(initializers[inp])
                if 'bias' in inp and inp in initializers:
                    layer_data["bias"] = initializers[inp].flatten().tolist()

        # Pooling Konfiguration
        if layer_type == 'MAX_POOLING':
            attrs = {attr.name: list(attr.ints) for attr in node.attribute}
            layer_data["config"] = {
                "kernel_size": attrs.get("kernel_shape", []),
                "stride": attrs.get("strides", [])
            }

        hardware_layers.append(layer_data)
        layer_idx += 1
        
    return hardware_layers

# Im Notebook ausführen:
fused_graph = parse_onnx_with_fused_relu("simple_mnist_convnet_int8.onnx")

# Übersicht ausgeben
print("--- Generierte Hardware-Pipeline (mit Fused ReLU) ---\n")
for layer in fused_graph:
    print(f"[{layer['id']}] Typ: {layer['type']:<15} | Name: {layer['name']:<25} | Fused ReLU: {layer['has_relu']}")

--- Generierte Hardware-Pipeline (mit Fused ReLU) ---

[1] Typ: CONVOLUTION     | Name: /Conv1/Conv_quant         | Fused ReLU: True
[2] Typ: MAX_POOLING     | Name: /Pool1/MaxPool            | Fused ReLU: False
[3] Typ: CONVOLUTION     | Name: /Conv2/Conv_quant         | Fused ReLU: True
[4] Typ: MAX_POOLING     | Name: /Pool2/MaxPool            | Fused ReLU: False
[5] Typ: CONVOLUTION     | Name: /Conv3/Conv_quant         | Fused ReLU: True
[6] Typ: FULLY_CONNECTED | Name: /FC1/Gemm_MatMul_quant    | Fused ReLU: False


In [56]:
import onnx
from onnx import numpy_helper

def parse_complete_onnx_pipeline(model_path):
    model = onnx.load(model_path)
    initializers = {init.name: numpy_helper.to_array(init) for init in model.graph.initializer}
    
    next_node_map = {}
    for node in model.graph.node:
        for inp in node.input:
            next_node_map[inp] = node

    hardware_layers = []
    layer_idx = 1
    
    for node in model.graph.node:
        if node.op_type not in ['ConvInteger', 'QLinearConv', 'MatMulInteger', 'QLinearMatMul', 'MaxPool']:
            continue
            
        if node.op_type in ['ConvInteger', 'QLinearConv']:
            layer_type = 'CONVOLUTION'
        elif node.op_type in ['MatMulInteger', 'QLinearMatMul']:
            layer_type = 'FULLY_CONNECTED'
        elif node.op_type == 'MaxPool':
            layer_type = 'MAX_POOLING'

        layer_data = {
            "id": layer_idx,
            "type": layer_type,
            "name": node.name,
            "has_relu": False,
            "hyperparameters": {},
            "weights": None,
            "bias": None,
            "scale": None,
            "zero_point": None
        }
        
        # --- LOOK-AHEAD LOGIK (ReLU) ---
        current_tensor = node.output[0]
        for _ in range(5):
            if current_tensor in next_node_map:
                next_node = next_node_map[current_tensor]
                if next_node.op_type == 'Relu':
                    layer_data["has_relu"] = True
                    break
                current_tensor = next_node.output[0]
            else:
                break

        # --- HYPERPARAMETER ---
        node_attrs = {}
        for attr in node.attribute:
            if attr.type == onnx.AttributeProto.INTS:
                node_attrs[attr.name] = list(attr.ints)
            elif attr.type == onnx.AttributeProto.INT:
                node_attrs[attr.name] = attr.i

        if layer_type == 'CONVOLUTION':
            layer_data["hyperparameters"] = {
                "kernel_shape": node_attrs.get("kernel_shape", [3, 3]),
                "strides": node_attrs.get("strides", [1, 1]),
                "padding_pads": node_attrs.get("pads", [0, 0, 0, 0]),
            }
        elif layer_type == 'MAX_POOLING':
            layer_data["hyperparameters"] = {
                "kernel_shape": node_attrs.get("kernel_shape", [2, 2]),
                "strides": node_attrs.get("strides", [2, 2]),
                "padding_pads": node_attrs.get("pads", [0, 0, 0, 0])
            }

        # --- GEWICHTE, BIAS, SCALE, ZERO-POINT EXTRAKTION ---
        if layer_type in ['CONVOLUTION', 'FULLY_CONNECTED']:
            # 1. Gewichte holen (Eingang Index 1)
            weight_name = node.input[1]
            if weight_name in initializers:
                w_array = initializers[weight_name]
                layer_data["weights"] = w_array
                
                # In/Out Channels direkt aus der Shape lesen und in Hyperparameter speichern
                if layer_type == 'CONVOLUTION':
                    layer_data["hyperparameters"]["out_channels"] = w_array.shape[0]
                    layer_data["hyperparameters"]["in_channels"] = w_array.shape[1]
                elif layer_type == 'FULLY_CONNECTED':
                    layer_data["hyperparameters"]["out_features"] = w_array.shape[0]
                    layer_data["hyperparameters"]["in_features"] = w_array.shape[1]

            # 2. Skalierungsfaktoren, Zero-Points und Biases einsammeln
            for inp in node.input:
                if inp in initializers:
                    if 'scale' in inp:
                        # Holt den float-Wert des Scales
                        layer_data["scale"] = float(initializers[inp])
                    elif 'zero_point' in inp:
                        # Holt den Integer-Wert des Nullpunkts
                        layer_data["zero_point"] = int(initializers[inp])
                    elif 'bias' in inp:
                        # Holt das Bias-Array (float32)
                        layer_data["bias"] = initializers[inp]

        hardware_layers.append(layer_data)
        layer_idx += 1
        
    return hardware_layers

# Inferenzpipeline komplett generieren
final_pipeline = parse_complete_onnx_pipeline("simple_mnist_convnet_int8.onnx")
print("Pipeline erfolgreich aufgebaut! Alle Gewichte und Parameter sind geladen.")

Pipeline erfolgreich aufgebaut! Alle Gewichte und Parameter sind geladen.
